# Phase 2 – Demographic and Equity Overlay

Join CCD, ACS, and EJScreen data. Test for environmental inequity using chi-square tests.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from src.demographic_overlay import run_phase2, equity_analysis
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import json

## Run Phase 2

In [ ]:
schools = run_phase2()
print(schools.columns.tolist())
schools.head()

## Equity Analysis

In [ ]:
results = equity_analysis(schools)
for k, v in results.items():
    print(f'{k}: chi2={v["chi2"]}, p={v["p"]}, dof={v["dof"]}')
    if v['p'] < 0.05:
        print('  --> Statistically significant: high-noise schools are NOT randomly distributed')

## FRL% vs. Noise Exposure

In [ ]:
frl_col = next((c for c in schools.columns if 'FRL' in c or 'FREE' in c), None)
if frl_col and 'noise_tier_label' in schools.columns:
    COLORS = {
        'Tier 1 - Minimal Risk': '#2ecc71',
        'Tier 2 - Elevated Risk': '#f1c40f',
        'Tier 3 - Significant Impact': '#e67e22',
        'Tier 4 - Severe Exposure': '#e74c3c',
    }
    schools[frl_col] = pd.to_numeric(schools[frl_col], errors='coerce')
    fig, ax = plt.subplots(figsize=(10, 6))
    for label, color in COLORS.items():
        sub = schools[schools['noise_tier_label'] == label]
        ax.scatter(sub[frl_col], sub['noise_db'], c=color, label=label, alpha=0.4, s=6)
    ax.axhline(55, color='white', linestyle='--', linewidth=1.5, label='WHO 55 dB')
    ax.set_xlabel('Free/Reduced Lunch Eligibility (%)')
    ax.set_ylabel('Noise Level (dB LAeq)')
    ax.set_title('School Poverty Rate vs. Noise Exposure')
    ax.legend(fontsize=8, markerscale=3)
    plt.tight_layout()
    plt.show()
else:
    print('FRL data not available.')

## State-Level High-Noise Proportion

In [ ]:
state_col = next((c for c in schools.columns if c in ('STABR','ST','STATE_ABBR','STABBR')), None)
if state_col:
    schools['high_noise'] = schools['noise_tier'].isin([3.0, 4.0]).astype(float)
    state_summary = (schools.groupby(state_col)['high_noise']
                     .agg(['mean','count']).reset_index())
    state_summary.columns = [state_col, 'pct_high_noise', 'n_schools']
    state_summary['pct_high_noise'] = (state_summary['pct_high_noise']*100).round(1)
    print(state_summary.sort_values('pct_high_noise', ascending=False).head(10))
else:
    print('No state column found.')